# 1강: NTFS MFT 파싱 실습

이 노트북에서는 NTFS Master File Table(MFT)의 구조를 이해하고,
Python ctypes를 사용하여 MFT 레코드를 직접 읽어보는 실습을 합니다.

> **⚠️ 주의**: 이 실습은 관리자 권한으로 실행해야 합니다.

## 1.1 MFT 기본 개념

MFT는 NTFS 볼륨의 핵심 메타데이터 구조입니다:
- 볼륨 내 **모든 파일과 디렉터리**에 대해 하나 이상의 레코드(기본 1024바이트)를 할당
- 레코드 0번 = `$MFT` (MFT 자체를 가리키는 메타 레코드)
- 각 레코드 내부에 **속성(attribute)**이 체인 형태로 배치

### 주요 속성 타입

| 타입 코드 | 이름 | 설명 |
|-----------|------|------|
| 0x10 | $STANDARD_INFORMATION | 생성일, 수정일, 접근일, 파일 플래그 |
| 0x30 | $FILE_NAME | 부모 디렉터리 참조, 파일명, 네임스페이스 |
| 0x80 | $DATA | 파일 본문 데이터 (resident 또는 non-resident) |

## 1.2 Windows API로 볼륨 핸들 열기

MFT에 접근하려면 `CreateFileW`로 볼륨을 직접 열어야 합니다.

In [ ]:
import ctypes
import ctypes.wintypes as wintypes

# Windows API 상수
GENERIC_READ           = 0x80000000
FILE_SHARE_READ        = 0x00000001
FILE_SHARE_WRITE       = 0x00000002
OPEN_EXISTING          = 3
FILE_FLAG_BACKUP_SEMANTICS = 0x02000000
INVALID_HANDLE_VALUE   = wintypes.HANDLE(-1).value

# kernel32 함수 바인딩
kernel32 = ctypes.windll.kernel32
CreateFileW = kernel32.CreateFileW
CreateFileW.restype = wintypes.HANDLE
CreateFileW.argtypes = [
    wintypes.LPCWSTR, wintypes.DWORD, wintypes.DWORD,
    ctypes.c_void_p, wintypes.DWORD, wintypes.DWORD, wintypes.HANDLE,
]

CloseHandle = kernel32.CloseHandle

# C:\ 볼륨 핸들 열기
handle = CreateFileW(
    r'\\.\C:',
    GENERIC_READ,
    FILE_SHARE_READ | FILE_SHARE_WRITE,
    None,
    OPEN_EXISTING,
    FILE_FLAG_BACKUP_SEMANTICS,
    None,
)

if handle == INVALID_HANDLE_VALUE:
    print(f'볼륨 열기 실패 (오류 코드: {kernel32.GetLastError()})')
    print('관리자 권한으로 실행했는지 확인하세요.')
else:
    print(f'볼륨 핸들 획득 성공: {handle}')

## 1.3 NTFS 볼륨 메타데이터 조회

`FSCTL_GET_NTFS_VOLUME_DATA`로 MFT 레코드 크기, 클러스터 크기 등을 알 수 있습니다.

In [ ]:
FSCTL_GET_NTFS_VOLUME_DATA = 0x00090064

DeviceIoControl = kernel32.DeviceIoControl
DeviceIoControl.restype = wintypes.BOOL
DeviceIoControl.argtypes = [
    wintypes.HANDLE, wintypes.DWORD, ctypes.c_void_p, wintypes.DWORD,
    ctypes.c_void_p, wintypes.DWORD, ctypes.POINTER(wintypes.DWORD),
    ctypes.c_void_p,
]

class NTFS_VOLUME_DATA_BUFFER(ctypes.Structure):
    _fields_ = [
        ('VolumeSerialNumber',            ctypes.c_longlong),
        ('NumberSectors',                 ctypes.c_longlong),
        ('TotalClusters',                 ctypes.c_longlong),
        ('FreeClusters',                  ctypes.c_longlong),
        ('TotalReserved',                 ctypes.c_longlong),
        ('BytesPerSector',                wintypes.DWORD),
        ('BytesPerCluster',               wintypes.DWORD),
        ('BytesPerFileRecordSegment',     wintypes.DWORD),
        ('ClustersPerFileRecordSegment',  wintypes.DWORD),
        ('MftValidDataLength',            ctypes.c_longlong),
        ('MftStartLcn',                   ctypes.c_longlong),
        ('Mft2StartLcn',                  ctypes.c_longlong),
        ('MftZoneStart',                  ctypes.c_longlong),
        ('MftZoneEnd',                    ctypes.c_longlong),
    ]

vol_data = NTFS_VOLUME_DATA_BUFFER()
bytes_returned = wintypes.DWORD(0)
ok = DeviceIoControl(
    handle, FSCTL_GET_NTFS_VOLUME_DATA,
    None, 0,
    ctypes.byref(vol_data), ctypes.sizeof(vol_data),
    ctypes.byref(bytes_returned), None,
)

if ok:
    print(f'MFT 레코드 크기:     {vol_data.BytesPerFileRecordSegment} 바이트')
    print(f'바이트/섹터:         {vol_data.BytesPerSector}')
    print(f'바이트/클러스터:     {vol_data.BytesPerCluster}')
    print(f'MFT 시작 LCN:       {vol_data.MftStartLcn}')
    print(f'MFT 유효 데이터:     {vol_data.MftValidDataLength:,} 바이트')
    print(f'총 MFT 레코드 수:   ~{vol_data.MftValidDataLength // vol_data.BytesPerFileRecordSegment:,}개')
else:
    print(f'NTFS 볼륨 데이터 조회 실패: {kernel32.GetLastError()}')

## 1.4 USN Journal 조회

USN Journal은 파일 시스템 변경 로그입니다. `FSCTL_QUERY_USN_JOURNAL`로 상태를 확인합니다.

In [ ]:
FSCTL_QUERY_USN_JOURNAL = 0x000900F4

class USN_JOURNAL_DATA(ctypes.Structure):
    _fields_ = [
        ('UsnJournalID',    ctypes.c_ulonglong),
        ('FirstUsn',        ctypes.c_longlong),
        ('NextUsn',         ctypes.c_longlong),
        ('LowestValidUsn',  ctypes.c_longlong),
        ('MaxUsn',          ctypes.c_longlong),
        ('MaximumSize',     ctypes.c_ulonglong),
        ('AllocationDelta', ctypes.c_ulonglong),
    ]

journal = USN_JOURNAL_DATA()
ok = DeviceIoControl(
    handle, FSCTL_QUERY_USN_JOURNAL,
    None, 0,
    ctypes.byref(journal), ctypes.sizeof(journal),
    ctypes.byref(bytes_returned), None,
)

if ok:
    print(f'Journal ID:       {journal.UsnJournalID}')
    print(f'Next USN:         {journal.NextUsn:,}')
    print(f'Lowest Valid USN: {journal.LowestValidUsn:,}')
    print(f'Maximum Size:     {journal.MaximumSize:,} 바이트')
else:
    print(f'USN Journal 조회 실패: {kernel32.GetLastError()}')

## 1.5 FILETIME → Python datetime 변환

NTFS의 시간은 Windows FILETIME 형식(1601-01-01 UTC 기준 100나노초 단위)으로 저장됩니다.

In [ ]:
from datetime import datetime, timezone

# FILETIME 에포크 차이 (1601-01-01 ~ 1970-01-01) 의 100ns 틱 수
FT_EPOCH_DIFF = 116444736000000000
FT_TICKS_SEC  = 10_000_000

def filetime_to_datetime(ft: int) -> datetime | None:
    """Windows FILETIME (100ns 단위) → Python datetime 변환"""
    if ft <= FT_EPOCH_DIFF:
        return None
    unix_ts = (ft - FT_EPOCH_DIFF) / FT_TICKS_SEC
    return datetime.fromtimestamp(unix_ts)

# 테스트: 현재 시각을 FILETIME으로 변환 → 다시 datetime으로
import time
now_ft = int(time.time() * FT_TICKS_SEC) + FT_EPOCH_DIFF
print(f'현재 FILETIME 값:  {now_ft}')
print(f'변환된 datetime: {filetime_to_datetime(now_ft)}')

## 1.6 SeekSeek의 MFT 스캐너 사용해보기

SeekSeek의 `core.mft_scanner` 모듈을 직접 호출하여 MFT를 열거해봅시다.

In [ ]:
import sys, os
# 프로젝트 루트를 경로에 추가
sys.path.insert(0, os.path.abspath('..'))

from core.mft_scanner import get_ntfs_drives, enumerate_mft

# 시스템의 NTFS 드라이브 확인
drives = get_ntfs_drives()
print(f'NTFS 드라이브: {drives}')

# C: 드라이브 MFT 열거 (시간이 걸릴 수 있음)
if 'C' in drives:
    result = enumerate_mft('C')
    print(f'성공: {result.success}')
    print(f'파일 수: {result.total_entries:,}')
    print(f'Journal ID: {result.journal_id}')
    print(f'Next USN: {result.next_usn}')
    # 처음 10개 파일 출력
    for entry in result.files[:10]:
        print(f'  {entry.full_path}  ({entry.size:,} bytes)')

In [ ]:
# 정리: 볼륨 핸들 닫기
CloseHandle(handle)
print('볼륨 핸들 닫기 완료')

## 요약

- `CreateFileW(r'\\.\C:')` — 볼륨 핸들 열기 (관리자 필수)
- `DeviceIoControl(FSCTL_GET_NTFS_VOLUME_DATA)` — MFT 메타데이터 조회
- `DeviceIoControl(FSCTL_QUERY_USN_JOURNAL)` — USN Journal 상태 조회
- FILETIME은 1601-01-01 기준 100ns 단위 → Unix timestamp 변환 필요
- SeekSeek의 `enumerate_mft()`은 MFT 직접 파싱 → USN 폴백 2단계 전략